# Random Forest SiPM Localization

Train a tabular random-forest regressor on the 32 SiPM-count vector and predict continuous `(x, z)` hit position for the 1000 random single-muon test events.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "macros").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAINING_DIR = PROJECT_ROOT / "analysis/training_scan_data_1024"
TEST_DIR = PROJECT_ROOT / "analysis/test_scan_data_1000"
TEST_MANIFEST = TEST_DIR / "test_manifest.csv"
MACRO_PATH = PROJECT_ROOT / "macros/muon_scan.mac"

SUMMARY_CSV = PROJECT_ROOT / "analysis/random_forest_sipm_summary.csv"
PREDICTIONS_CSV = PROJECT_ROOT / "analysis/random_forest_sipm_predictions.csv"
ERROR_PNG = PROJECT_ROOT / "analysis/random_forest_sipm_error_map.png"

RUN_FILE_RE = re.compile(r"MUON-run(?P<run>\d+)_sipm_counts_by_event\.csv$")
SIPM_COLUMNS = [f"sipm_{i}" for i in range(100, 116)] + [f"sipm_{i}" for i in range(300, 316)]
RANDOM_STATE = 20260505

## Load Data

In [ ]:
def parse_muon_scan_positions(path: Path) -> pd.DataFrame:
    rows = []
    for line in path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) == 5 and parts[0] == "/gps/position":
            x_cm, _, z_cm = map(float, parts[1:4])
            rows.append({"run_id": len(rows), "x_cm": x_cm, "z_cm": z_cm})
    if not rows:
        raise ValueError(f"No /gps/position lines found in {path}")
    return pd.DataFrame(rows)


def load_training_events(path: Path) -> pd.DataFrame:
    frames = []
    for csv_path in sorted(path.glob("MUON-run*_sipm_counts_by_event.csv")):
        match = RUN_FILE_RE.match(csv_path.name)
        if not match:
            continue
        df = pd.read_csv(csv_path)
        missing = [col for col in ["EventID", *SIPM_COLUMNS] if col not in df.columns]
        if missing:
            raise KeyError(f"{csv_path} is missing columns: {missing}")
        df = df[["EventID", *SIPM_COLUMNS]].copy()
        df.insert(0, "run_id", int(match.group("run")))
        df = df.rename(columns={"EventID": "event_id"})
        frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No training CSV files found in {path}")
    return pd.concat(frames, ignore_index=True)


def load_test_events(test_dir: Path, manifest_path: Path) -> pd.DataFrame:
    manifest = pd.read_csv(manifest_path)
    rows = []
    for _, meta in manifest.sort_values("event_id").iterrows():
        df = pd.read_csv(test_dir / meta["output_file"])
        if len(df) != 1:
            raise ValueError(f"Expected one row in {meta['output_file']}, found {len(df)}")
        row = df.iloc[0].to_dict()
        row["event_id"] = int(meta["event_id"])
        row["run_id"] = int(meta["run_id"])
        row["raw_file"] = meta["raw_file"]
        row["true_x_cm"] = float(meta["true_x_cm"])
        row["true_z_cm"] = float(meta["true_z_cm"])
        rows.append(row)
    return pd.DataFrame(rows).sort_values("event_id").reset_index(drop=True)


labels = parse_muon_scan_positions(MACRO_PATH)
train_events = load_training_events(TRAINING_DIR).merge(labels, on="run_id", validate="many_to_one")
test_events = load_test_events(TEST_DIR, TEST_MANIFEST)

X_train = train_events[SIPM_COLUMNS].to_numpy(dtype=np.float64)
y_train = train_events[["x_cm", "z_cm"]].to_numpy(dtype=np.float64)
X_test = test_events[SIPM_COLUMNS].to_numpy(dtype=np.float64)
y_test = test_events[["true_x_cm", "true_z_cm"]].to_numpy(dtype=np.float64)

print(f"training events: {len(train_events)}")
print(f"test events: {len(test_events)}")

## Train a Few Random-Forest Variants

The target is continuous `(x, z)` in cm. We try raw counts and `log1p` counts, with mildly regularized leaves to reduce single-event noise.

In [ ]:
def identity(x):
    return x


def log1p_nonnegative(x):
    return np.log1p(np.maximum(x, 0.0))


configs = [
    {
        "name": "rf_raw_leaf1",
        "transform": identity,
        "n_estimators": 300,
        "min_samples_leaf": 1,
        "max_features": 1.0,
    },
    {
        "name": "rf_raw_leaf3",
        "transform": identity,
        "n_estimators": 300,
        "min_samples_leaf": 3,
        "max_features": 1.0,
    },
    {
        "name": "rf_log_leaf3",
        "transform": log1p_nonnegative,
        "n_estimators": 300,
        "min_samples_leaf": 3,
        "max_features": 1.0,
    },
    {
        "name": "rf_log_leaf5_sqrt_features",
        "transform": log1p_nonnegative,
        "n_estimators": 300,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
    },
]


def make_model(config):
    return make_pipeline(
        FunctionTransformer(config["transform"], validate=False),
        RandomForestRegressor(
            n_estimators=config["n_estimators"],
            min_samples_leaf=config["min_samples_leaf"],
            max_features=config["max_features"],
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbose=0,
        ),
    )


def evaluate_predictions(pred, config_name):
    err = pred - y_test
    err_r = np.hypot(err[:, 0], err[:, 1])
    return {
        "model": config_name,
        "n_events": len(pred),
        "mean_err_x_cm": float(err[:, 0].mean()),
        "sigma_err_x_cm": float(err[:, 0].std(ddof=1)),
        "mean_err_z_cm": float(err[:, 1].mean()),
        "sigma_err_z_cm": float(err[:, 1].std(ddof=1)),
        "median_err_r_cm": float(np.median(err_r)),
        "p68_err_r_cm": float(np.quantile(err_r, 0.68)),
        "p95_err_r_cm": float(np.quantile(err_r, 0.95)),
    }


summary_rows = []
prediction_store = {}

for config in configs:
    print(f"training {config['name']}...")
    model = make_model(config)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prediction_store[config["name"]] = pred
    summary_rows.append(evaluate_predictions(pred, config["name"]))

summary = pd.DataFrame(summary_rows).sort_values("p68_err_r_cm").reset_index(drop=True)
summary

## Save Best Result

In [ ]:
best_name = summary.iloc[0]["model"]
best_pred = prediction_store[best_name]
results = test_events[["event_id", "run_id", "raw_file", "true_x_cm", "true_z_cm", *SIPM_COLUMNS]].copy()
results["pred_x_cm"] = best_pred[:, 0]
results["pred_z_cm"] = best_pred[:, 1]
results["err_x_cm"] = results["pred_x_cm"] - results["true_x_cm"]
results["err_z_cm"] = results["pred_z_cm"] - results["true_z_cm"]
results["err_r_cm"] = np.hypot(results["err_x_cm"], results["err_z_cm"])
results["model"] = best_name

summary.to_csv(SUMMARY_CSV, index=False)
results.to_csv(PREDICTIONS_CSV, index=False)

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(results["true_x_cm"], results["true_z_cm"], c=results["err_r_cm"], s=18, cmap="viridis")
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("true x [cm]")
ax.set_ylabel("true z [cm]")
ax.set_title(f"Random forest radial error: {best_name}")
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("radial error [cm]")
fig.tight_layout()
fig.savefig(ERROR_PNG, dpi=180)

print(f"best model: {best_name}")
print(f"wrote {SUMMARY_CSV}")
print(f"wrote {PREDICTIONS_CSV}")
print(f"wrote {ERROR_PNG}")
summary